In [1]:
# import
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import json
from datetime import datetime, timedelta
from dateutil import parser

In [2]:
# 설정
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

MAX_PAGES = 5
BASE_DIR = "inven_out"

# 실제 게시판 URL
LOSTARK_MOBILE_URL = "https://www.inven.co.kr/board/lostarkmobile/6389"
AION2_URL = "https://www.inven.co.kr/board/aion2/6388?my=chuchu"

# 아이온2 출시 기준 2주
AION2_RELEASE_DATE = datetime(2025, 11, 19)
AION2_START = AION2_RELEASE_DATE - timedelta(days=14)
AION2_END = AION2_RELEASE_DATE + timedelta(days=14)

def sleep():
    time.sleep(random.uniform(1.0, 2.0))

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def load_state(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"done_urls": [], "data": []}

def save_state(path, state):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

In [3]:
# 링크 수집
def get_post_links(base_url, page):
    url = f"{base_url}?p={page}"
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    links = []
    rows = soup.select("table tbody tr")
    for row in rows:
        a_tag = row.select_one("td.tit a")
        if not a_tag:
            continue
        title = a_tag.text.strip()
        link = a_tag.get("href")
        links.append((title, link))
    return links

# 상세 수집
def get_post_detail(title, url):
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    content_tag = soup.select_one(".contentBody")
    content = content_tag.text.strip() if content_tag else ""

    created_at = ""
    date_tag = soup.select_one(".articleDate")
    if date_tag:
        created_at = date_tag.text.strip()

    views = 0
    view_tag = soup.select_one(".articleHit")
    if view_tag:
        try:
            views = int(view_tag.text.replace(",", "").strip())
        except:
            pass

    likes = 0
    like_tag = soup.select_one("div.recommend span.num")
    if like_tag:
        try:
            likes = int(like_tag.text.strip())
        except:
            pass

    comments = []
    comment_tags = soup.select(".cmtContent")
    for c in comment_tags:
        text = c.text.strip()
        if text:
            comments.append(text)

    return {
        "title": title,
        "content": content,
        "comments": comments,
        "url": url,
        "created_at": created_at,
        "views": views,
        "likes": likes,
        "comment_count": len(comments)
    }

# 크롤링 (중간 저장 + 진행률 표시)
def crawl_inven(base_url, game_type, filter_dates=None):
    game_dir = os.path.join(BASE_DIR, game_type)
    ensure_dir(game_dir)

    state_path = os.path.join(game_dir, "state.json")
    state = load_state(state_path)

    done_urls = set(state["done_urls"])
    data = state["data"]

    all_links = []
    for page in range(1, MAX_PAGES + 1):
        print(f"[{game_type}] page {page}/{MAX_PAGES}")
        links = get_post_links(base_url, page)
        all_links.extend(links)
        sleep()

    total_posts = len(all_links)
    print(f"[{game_type}] total {total_posts} posts")

    for idx, (title, link) in enumerate(all_links, 1):
        if link in done_urls:
            continue

        try:
            post_data = get_post_detail(title, link)

            # 아이온2 날짜 필터
            if filter_dates:
                try:
                    post_dt = datetime.strptime(post_data['created_at'], "%Y-%m-%d %H:%M")
                    if post_dt < filter_dates[0] or post_dt > filter_dates[1]:
                        continue
                except:
                    pass

            # 진행률 표시
            percent = round(idx / total_posts * 100, 1)
            print(f"[posts] {idx}/{total_posts} ({percent}%)")
            print(f"[comments] {link.split('/')[-1]} -> {post_data['comment_count']} comments")

            data.append(post_data)
            done_urls.add(link)

            # 중간저장
            save_state(state_path, {"done_urls": list(done_urls), "data": data})
            sleep()
        except Exception as e:
            print("error:", e)

    return data

# CSV 저장
def save_to_csv(data, game_type):
    game_dir = os.path.join(BASE_DIR, game_type)
    ensure_dir(game_dir)

    today = datetime.now().strftime("%Y%m%d")
    filename = os.path.join(game_dir, f"{game_type}_{today}.csv")

    rows = []
    for post in data:
        if post["comments"]:
            for c in post["comments"]:
                rows.append({
                    "title": post["title"],
                    "content": post["content"],
                    "comment": c,
                    "url": post["url"],
                    "created_at": post["created_at"],
                    "views": post["views"],
                    "likes": post["likes"],
                    "comment_count": post["comment_count"]
                })
        else:
            rows.append({
                "title": post["title"],
                "content": post["content"],
                "comment": "",
                "url": post["url"],
                "created_at": post["created_at"],
                "views": post["views"],
                "likes": post["likes"],
                "comment_count": post["comment_count"]
            })

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved {filename}")

In [18]:
# 실행

# 로스트아크 모바일
lostark_data = crawl_inven(LOSTARK_MOBILE_URL, "lostark_mobile")
save_to_csv(lostark_data, "lostark_mobile")

# 아이온2
aion2_data = crawl_inven(
    AION2_URL,
    "aion2",
    filter_dates=(AION2_START, AION2_END)
)
save_to_csv(aion2_data, "aion2")

print("Done crawling")

[lostark_mobile] page 1/5
[lostark_mobile] page 2/5
[lostark_mobile] page 3/5
[lostark_mobile] page 4/5
[lostark_mobile] page 5/5
[lostark_mobile] total 226 posts
[posts] 1/226 (0.4%)
[comments] 1176 -> 0 comments
[posts] 2/226 (0.9%)
[comments] 1229 -> 0 comments
[posts] 3/226 (1.3%)
[comments] 1197 -> 0 comments
[posts] 4/226 (1.8%)
[comments] 1193 -> 0 comments
[posts] 5/226 (2.2%)
[comments] 1191 -> 0 comments
[posts] 6/226 (2.7%)
[comments] 1190 -> 0 comments
[posts] 7/226 (3.1%)
[comments] 1189 -> 0 comments
[posts] 8/226 (3.5%)
[comments] 1188 -> 0 comments
[posts] 9/226 (4.0%)
[comments] 1187 -> 0 comments
[posts] 10/226 (4.4%)
[comments] 1186 -> 0 comments
[posts] 11/226 (4.9%)
[comments] 1185 -> 0 comments
[posts] 12/226 (5.3%)
[comments] 1184 -> 0 comments
[posts] 13/226 (5.8%)
[comments] 1181 -> 0 comments
[posts] 14/226 (6.2%)
[comments] 1180 -> 0 comments
[posts] 15/226 (6.6%)
[comments] 1179 -> 0 comments
[posts] 16/226 (7.1%)
[comments] 1178 -> 0 comments
[posts] 17/226

In [4]:
import os
import json
import time
import random
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import requests
import pandas as pd
from bs4 import BeautifulSoup

In [11]:
"""===============================================
트러블 슈팅 : 아이온2 데이터 수집 오류 확인
원인 : 날짜 파싱 오류, 웹 크롤링 방식에서 URL 정상적으로 못 가져오는 현상, 인벤 게시판 구조가 다름

requests로 못 가져오고 브라우저 기반 수집(Selenium) 방법 선택
날짜 필터 유연화 (dateutil.parser)
중간 저장 (state.json) 적용
진행률 표시
CSV 저장
MAX_PAGES 확장 (2주치 게시글 확보)
==============================================="""
# import
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

BASE_URL = "https://www.inven.co.kr/board/aion2/6388"
HEADERS = {"User-Agent": "Mozilla/5.0"}

MAX_PAGE = 20
BASE_DIR = "inven_out/aion2"
os.makedirs(BASE_DIR, exist_ok=True)

def get_post_links(page):
    url = f"{BASE_URL}?page={page}"
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    posts = []
    rows = soup.select("tr")

    for row in rows:
        title_tag = row.select_one("td.tit a.subject-link")
        if not title_tag:
            continue

        posts.append({
            "title": title_tag.text.strip(),
            "url": title_tag["href"]
        })

    return posts

def get_post_detail(post):
    res = requests.get(post["url"], headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    content_tag = soup.select_one(".contentBody")
    content = content_tag.text.strip() if content_tag else ""

    # 댓글
    comments = []
    comment_tags = soup.select(".cmtContent")
    for c in comment_tags:
        txt = c.text.strip()
        if txt:
            comments.append(txt)

    return {
        "title": post["title"],
        "content": content,
        "comments": comments,
        "url": post["url"],
        "comment_count": len(comments)
    }

# 전체 수집
all_posts = []

for page in range(1, MAX_PAGE + 1):
    print(f"[게시글] {page}/{MAX_PAGE} 페이지 수집 중 ({round(page/MAX_PAGE*100,1)}%)")

    posts = get_post_links(page)
    if not posts:
        print("게시글 없음 → 종료")
        break

    all_posts.extend(posts)
    time.sleep(1)

print(f"총 게시글 수: {len(all_posts)}")

# 상세 + 댓글 수집
final_data = []

total = len(all_posts)

for idx, post in enumerate(all_posts, 1):
    print(f"[댓글] {idx}/{total} ({round(idx/total*100,1)}%) 수집 중")

    detail = get_post_detail(post)

    if detail["comments"]:
        for c in detail["comments"]:
            final_data.append({
                "title": detail["title"],
                "content": detail["content"],
                "comment": c,
                "url": detail["url"],
                "comment_count": detail["comment_count"]
            })
    else:
        final_data.append({
            "title": detail["title"],
            "content": detail["content"],
            "comment": "",
            "url": detail["url"],
            "comment_count": detail["comment_count"]
        })

    time.sleep(0.5)

# 저장 
df = pd.DataFrame(final_data)

today = pd.Timestamp.now().strftime("%Y%m%d")
filename = os.path.join(BASE_DIR, f"aion2_{today}.csv")

df.to_csv(filename, index=False, encoding="utf-8-sig")

print(f"저장 완료: {filename}")

[게시글] 1/20 페이지 수집 중 (5.0%)
[게시글] 2/20 페이지 수집 중 (10.0%)
[게시글] 3/20 페이지 수집 중 (15.0%)
[게시글] 4/20 페이지 수집 중 (20.0%)
[게시글] 5/20 페이지 수집 중 (25.0%)
[게시글] 6/20 페이지 수집 중 (30.0%)
[게시글] 7/20 페이지 수집 중 (35.0%)
[게시글] 8/20 페이지 수집 중 (40.0%)
[게시글] 9/20 페이지 수집 중 (45.0%)
[게시글] 10/20 페이지 수집 중 (50.0%)
[게시글] 11/20 페이지 수집 중 (55.0%)
[게시글] 12/20 페이지 수집 중 (60.0%)
[게시글] 13/20 페이지 수집 중 (65.0%)
[게시글] 14/20 페이지 수집 중 (70.0%)
[게시글] 15/20 페이지 수집 중 (75.0%)
[게시글] 16/20 페이지 수집 중 (80.0%)
[게시글] 17/20 페이지 수집 중 (85.0%)
[게시글] 18/20 페이지 수집 중 (90.0%)
[게시글] 19/20 페이지 수집 중 (95.0%)
[게시글] 20/20 페이지 수집 중 (100.0%)
총 게시글 수: 1400
[댓글] 1/1400 (0.1%) 수집 중
[댓글] 2/1400 (0.1%) 수집 중
[댓글] 3/1400 (0.2%) 수집 중
[댓글] 4/1400 (0.3%) 수집 중
[댓글] 5/1400 (0.4%) 수집 중
[댓글] 6/1400 (0.4%) 수집 중
[댓글] 7/1400 (0.5%) 수집 중
[댓글] 8/1400 (0.6%) 수집 중
[댓글] 9/1400 (0.6%) 수집 중
[댓글] 10/1400 (0.7%) 수집 중
[댓글] 11/1400 (0.8%) 수집 중
[댓글] 12/1400 (0.9%) 수집 중
[댓글] 13/1400 (0.9%) 수집 중
[댓글] 14/1400 (1.0%) 수집 중
[댓글] 15/1400 (1.1%) 수집 중
[댓글] 16/1400 (1.1%) 수집 중
[댓글] 17/1400 (1.2%) 수집 중

In [12]:
# 한 파일로 합치기

# 파일 경로
aion_path = "inven_out/aion2/aion2_20260326.csv"
lostark_path = "inven_out/lostark_mobile/lostark_mobile_20260325.csv"

# 불러오기
aion_df = pd.read_csv(aion_path)
lostark_df = pd.read_csv(lostark_path)

# 컬럼 추가
aion_df["source"] = "inven_aion2"
lostark_df["source"] = "inven_lostarkmobile"

# 공통 region
aion_df["region"] = "국내"
lostark_df["region"] = "국내"

# 컬럼 구조 맞추기 (혹시 컬럼 다를 경우 대비)
common_cols = list(set(aion_df.columns) | set(lostark_df.columns))

aion_df = aion_df.reindex(columns=common_cols)
lostark_df = lostark_df.reindex(columns=common_cols)

# 합치기
merged_df = pd.concat([aion_df, lostark_df], ignore_index=True)

print("합치기 완료")
print("총 데이터 수:", len(merged_df))
print("컬럼:", merged_df.columns.tolist())

합치기 완료
총 데이터 수: 1625
컬럼: ['region', 'created_at', 'comment_count', 'comment', 'url', 'views', 'title', 'source', 'content', 'likes']


In [13]:
merged_df

,region,created_at,comment_count,comment,url,views,title,source,content,likes
0,국내,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/72656,NaN,[잡담]\n아이온2 인장 캐릭터 인증 가이드,inven_aion2,아이온2 인벤에 아이온2 인장 연동 기능이 추가되었습니다. 아래 안내를 따라와 주시...,NaN
1,국내,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/104900,NaN,[소식]\n ...,inven_aion2,쿠폰 입력 기간 변경 [바로가기]시즌2 일정 변경에 따라 AION2 웰컴백 쿠폰 &...,NaN
2,국내,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/125736,NaN,"[잡담]\n전투력 업데이트 완료! 인장 갱신하면 3,000 이니 드려요!",inven_aion2,"안녕하세요, 아이온2 인벤팀입니다.아이온2 인증 인장에 전투력 수치가 추가되었습니다...",NaN
3,국내,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131176,NaN,[소식]\n ...,inven_aion2,[콘텐츠 및 시스템]어비스 매칭 개편어비스 매칭에서 동일 종족도 적대 종족으로 매칭...,NaN
4,국내,NaN,0,NaN,https://www.inven.co.kr/board/aion2/6388/131177,NaN,[소식]\n ...,inven_aion2,"두 개의 하늘, 하나의 영광안녕하세요, 데바 여러분. AION2입니다.​항상 AIO...",NaN
...,...,...,...,...,...,...,...,...,...,...
1620,국내,2025-11-14 16:47,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...,0.0,[잡담]\n ...,inven_lostarkmobile,소환사 제대로 나와서 즐길수 있었으면 좋겠어요,0.0
1621,국내,2025-11-14 16:42,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...,0.0,[잡담]\n ...,inven_lostarkmobile,맞으며,0.0
1622,국내,2025-11-14 15:55,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...,0.0,[잡담]\n ...,inven_lostarkmobile,일단 나,0.0
1623,국내,2025-11-14 15:43,0,NaN,https://www.inven.co.kr/board/lostarkmobile/63...,0.0,[잡담]\n ...,inven_lostarkmobile,기대중입니다,0.0


In [ ]:
# 저장
merged_df.to_csv("inven_all_posts.csv", index=False, encoding="utf-8-sig")